# Agent 2 — Notebook 04 FINAL

## Approved Question Embeddings and Qdrant Indexing with MiniLM

This notebook indexes the Human-in-the-Loop-approved assessment questions
produced by Notebooks 01–03.

Embedding model:

```text
sentence-transformers/all-MiniLM-L6-v2
```

Vector configuration:

```text
dimensions = 384
distance   = cosine
```

Only retrieval-safe scored questions are indexed:

```sql
record_type = 'scored_item'
review_status IN ('human_approved', 'human_corrected')
retrieval_enabled = TRUE
is_active = TRUE
is_legacy = FALSE
embedding_status IN ('ready_for_indexing', 'indexed')
```

Current expected eligible count after Notebook 03:

```text
820 scored questions
```

Context-only, rejected, and legacy questions are never indexed.


## Pipeline

```text
PostgreSQL approved questions
            ↓
build retrieval-focused text
            ↓
MiniLM embeddings (384 dimensions)
            ↓
Qdrant collection + payload indexes
            ↓
idempotent point upsert
            ↓
PostgreSQL vector-index metadata
            ↓
semantic-search and self-retrieval checks
```


## 1. Install dependencies


In [49]:
%pip install -q "sentence-transformers>=3.0,<6" "qdrant-client>=1.12,<2" pandas numpy torch python-dotenv "sqlalchemy>=2.0" "psycopg[binary]>=3.1"


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuration — Local Docker Qdrant

This notebook uses the same local Docker Qdrant server as Agent 1:

```text
QDRANT_URL=http://localhost:6333
QDRANT_API_KEY=
QDRANT_TIMEOUT_SECONDS=30
QDRANT_TOP_K_PER_UNIT=20
QDRANT_SCORE_THRESHOLD=
```

Agent 2 should use a **separate collection** so syllabus/topic vectors from Agent 1
are not mixed with assessment-question vectors.

Add this extra value to `.env`:

```text
AGENT2_QDRANT_COLLECTION=aqa_gcse_computer_science_8525_questions
```

If `AGENT2_QDRANT_COLLECTION` is missing, the notebook safely derives:

```text
<QDRANT_COLLECTION>_questions
```

So your existing Agent 1 value:

```text
QDRANT_COLLECTION=aqa_gcse_computer_science_8525
```

becomes:

```text
aqa_gcse_computer_science_8525_questions
```

No Qdrant API key is required for your local Docker instance.

### Recommended execution order

First run:

```python
RUN_MODE = "pilot"
COMMIT_TO_QDRANT = False
```

Full dry run:

```python
RUN_MODE = "full"
COMMIT_TO_QDRANT = False
```

Final indexing run:

```python
RUN_MODE = "full"
COMMIT_TO_QDRANT = True
```


In [50]:
from __future__ import annotations

import hashlib
import json
import math
import os
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from IPython.display import display
from qdrant_client import QdrantClient, models
from qdrant_client.http.exceptions import UnexpectedResponse
from sentence_transformers import SentenceTransformer
from sqlalchemy import (
    Boolean,
    Column,
    DateTime,
    ForeignKey,
    Index,
    Integer,
    MetaData,
    String,
    Table,
    Text,
    UniqueConstraint,
    and_,
    create_engine,
    delete,
    func,
    insert,
    select,
    update,
)
from sqlalchemy.dialects.postgresql import JSONB, UUID, insert as pg_insert
from sqlalchemy.engine import Engine
from sqlalchemy.orm import Session


cwd = Path.cwd().resolve()

PROJECT_ROOT = (
    cwd.parent
    if cwd.name.lower() in {"notebooks", "notebook"}
    else cwd
)

OUTPUT_DIR = PROJECT_ROOT / "OUTPUT"
MODEL_CACHE_DIR = PROJECT_ROOT / "cache" / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")


DATABASE_URL = os.getenv(
    "AGENT2_DATABASE_URL",
    "",
).strip()

if not DATABASE_URL:
    raise RuntimeError(
        "AGENT2_DATABASE_URL is missing from Agent2/.env"
    )


QDRANT_URL = (
    os.getenv("QDRANT_URL")
    or "http://localhost:6333"
).strip()

# Local Docker Qdrant does not require an API key.
QDRANT_API_KEY = (
    os.getenv("QDRANT_API_KEY", "").strip()
    or None
)

AGENT1_COLLECTION_NAME = os.getenv(
    "QDRANT_COLLECTION",
    "aqa_gcse_computer_science_8525",
).strip()

COLLECTION_NAME = (
    os.getenv("AGENT2_QDRANT_COLLECTION", "").strip()
    or f"{AGENT1_COLLECTION_NAME}_questions"
)

QDRANT_TOP_K_PER_UNIT = int(
    os.getenv("QDRANT_TOP_K_PER_UNIT", "20")
)

QDRANT_TIMEOUT_SECONDS = int(
    os.getenv("QDRANT_TIMEOUT_SECONDS", "30")
)

_raw_score_threshold = os.getenv(
    "QDRANT_SCORE_THRESHOLD",
    "",
).strip()

QDRANT_SCORE_THRESHOLD = (
    float(_raw_score_threshold)
    if _raw_score_threshold
    else None
)


MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EXPECTED_VECTOR_SIZE = 384
EXPECTED_CURRENT_ELIGIBLE_COUNT = 820

RUN_MODE = "full"       # "pilot" or "full"
PILOT_SIZE = 32

COMMIT_TO_QDRANT = True
FORCE_REINDEX = False
RECREATE_COLLECTION = False
DELETE_STALE_POINTS = True

EMBEDDING_BATCH_SIZE = 32
QDRANT_UPSERT_BATCH_SIZE = 64
QUERY_LIMIT = QDRANT_TOP_K_PER_UNIT
RANDOM_SEED = 42

PREFER_GRPC = (
    os.getenv("QDRANT_PREFER_GRPC", "false")
    .strip()
    .lower()
    in {"1", "true", "yes"}
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

INDEXER_VERSION = "agent2-minilm-qdrant-indexer-v1.1.0-docker"

if RUN_MODE not in {"pilot", "full"}:
    raise ValueError(
        "RUN_MODE must be 'pilot' or 'full'."
    )

print(f"Project root:       {PROJECT_ROOT}")
print(f"Model:              {MODEL_NAME}")
print(f"Device:             {DEVICE}")
print(f"Qdrant URL:         {QDRANT_URL}")
print(f"API key configured: {QDRANT_API_KEY is not None}")
print(f"Agent 1 collection: {AGENT1_COLLECTION_NAME}")
print(f"Agent 2 collection: {COLLECTION_NAME}")
print(f"Top K per unit:     {QDRANT_TOP_K_PER_UNIT}")
print(f"Score threshold:    {QDRANT_SCORE_THRESHOLD}")
print(f"Timeout seconds:    {QDRANT_TIMEOUT_SECONDS}")
print(f"Run mode:           {RUN_MODE}")
print(f"Commit to Qdrant:   {COMMIT_TO_QDRANT}")
print(f"Force reindex:      {FORCE_REINDEX}")


Project root:       C:\Users\hp\EDTECH\Agent2
Model:              sentence-transformers/all-MiniLM-L6-v2
Device:             cpu
Qdrant URL:         http://localhost:6333
API key configured: False
Agent 1 collection: aqa_gcse_computer_science_8525
Agent 2 collection: aqa_gcse_computer_science_8525_questions
Top K per unit:     20
Score threshold:    None
Timeout seconds:    30
Run mode:           full
Commit to Qdrant:   True
Force reindex:      False


## Local Docker Qdrant preflight

Before continuing, make sure the container is running:

```powershell
docker ps
```

Your Qdrant dashboard should be available at:

```text
http://localhost:6333/dashboard
```

The following cell checks the local REST endpoint before loading MiniLM.


In [51]:
from urllib.error import URLError
from urllib.request import urlopen

qdrant_health_url = f"{QDRANT_URL.rstrip('/')}/collections"

try:
    with urlopen(
        qdrant_health_url,
        timeout=QDRANT_TIMEOUT_SECONDS,
    ) as response:
        qdrant_http_status = response.status

except URLError as error:
    raise RuntimeError(
        "Local Qdrant is not reachable. Start the Docker container "
        f"and confirm {QDRANT_URL} is accessible. Original error: {error}"
    ) from error

print(
    f"Local Qdrant preflight passed: "
    f"HTTP {qdrant_http_status}"
)


Local Qdrant preflight passed: HTTP 200


## 3. Connect to PostgreSQL and reflect approved assessment tables


In [52]:
engine: Engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,
    future=True,
)

metadata = MetaData()

topics = Table(
    "assessment_topical_topics",
    metadata,
    autoload_with=engine,
)

documents = Table(
    "assessment_topical_documents",
    metadata,
    autoload_with=engine,
)

questions = Table(
    "assessment_topical_questions",
    metadata,
    autoload_with=engine,
)

mark_scheme_entries = Table(
    "assessment_topical_mark_scheme_entries",
    metadata,
    autoload_with=engine,
)

question_ms_links = Table(
    "assessment_topical_question_mark_scheme_links",
    metadata,
    autoload_with=engine,
)

with engine.connect() as connection:
    connection.exec_driver_sql("SELECT 1")

print("PostgreSQL connection successful.")


PostgreSQL connection successful.


## 4. Create PostgreSQL vector-index audit tables

These tables make indexing incremental and auditable:

```text
assessment_embedding_runs
assessment_question_vector_index
```


In [53]:
embedding_runs = Table(
    "assessment_embedding_runs",
    metadata,
    Column(
        "id",
        UUID(as_uuid=True),
        primary_key=True,
        default=uuid.uuid4,
    ),
    Column(
        "indexer_version",
        String(120),
        nullable=False,
    ),
    Column(
        "embedding_model",
        String(200),
        nullable=False,
    ),
    Column(
        "embedding_dimension",
        Integer,
        nullable=False,
    ),
    Column(
        "collection_name",
        String(200),
        nullable=False,
    ),
    Column(
        "run_mode",
        String(20),
        nullable=False,
    ),
    Column(
        "committed",
        Boolean,
        nullable=False,
    ),
    Column(
        "status",
        String(40),
        nullable=False,
    ),
    Column(
        "source_count",
        Integer,
        nullable=False,
        default=0,
    ),
    Column(
        "embedded_count",
        Integer,
        nullable=False,
        default=0,
    ),
    Column(
        "indexed_count",
        Integer,
        nullable=False,
        default=0,
    ),
    Column(
        "skipped_count",
        Integer,
        nullable=False,
        default=0,
    ),
    Column(
        "failed_count",
        Integer,
        nullable=False,
        default=0,
    ),
    Column(
        "deleted_stale_count",
        Integer,
        nullable=False,
        default=0,
    ),
    Column(
        "configuration",
        JSONB,
        nullable=False,
        default=dict,
    ),
    Column(
        "started_at",
        DateTime(timezone=True),
        nullable=False,
    ),
    Column(
        "completed_at",
        DateTime(timezone=True),
    ),
    Column(
        "error_message",
        Text,
    ),
    Index(
        "ix_assessment_embedding_runs_started_at",
        "started_at",
    ),
)


question_vector_index = Table(
    "assessment_question_vector_index",
    metadata,
    Column(
        "id",
        UUID(as_uuid=True),
        primary_key=True,
        default=uuid.uuid4,
    ),
    Column(
        "question_id",
        UUID(as_uuid=True),
        ForeignKey(
            "assessment_topical_questions.id",
            ondelete="CASCADE",
        ),
        nullable=False,
    ),
    Column(
        "run_id",
        UUID(as_uuid=True),
        ForeignKey(
            "assessment_embedding_runs.id",
            ondelete="SET NULL",
        ),
    ),
    Column(
        "qdrant_point_id",
        String(100),
        nullable=False,
    ),
    Column(
        "collection_name",
        String(200),
        nullable=False,
    ),
    Column(
        "embedding_model",
        String(200),
        nullable=False,
    ),
    Column(
        "embedding_dimension",
        Integer,
        nullable=False,
    ),
    Column(
        "embedding_text_hash",
        String(64),
        nullable=False,
    ),
    Column(
        "payload_hash",
        String(64),
        nullable=False,
    ),
    Column(
        "vector_status",
        String(40),
        nullable=False,
    ),
    Column(
        "indexed_at",
        DateTime(timezone=True),
    ),
    Column(
        "last_verified_at",
        DateTime(timezone=True),
    ),
    Column(
        "error_message",
        Text,
    ),
    Column(
        "created_at",
        DateTime(timezone=True),
        nullable=False,
    ),
    Column(
        "updated_at",
        DateTime(timezone=True),
        nullable=False,
    ),
    UniqueConstraint(
        "question_id",
        "collection_name",
        name="uq_question_vector_index_question_collection",
    ),
    Index(
        "ix_question_vector_index_status",
        "collection_name",
        "vector_status",
    ),
)


metadata.create_all(
    engine,
    tables=[
        embedding_runs,
        question_vector_index,
    ],
)

print("Vector-index audit tables created/verified.")


Vector-index audit tables created/verified.


## 5. Load the retrieval-safe scored-question dataset


In [54]:
eligible_query = (
    select(
        questions.c.id.label("question_id"),
        questions.c.question_uid,
        questions.c.pair_key,
        questions.c.topic_id,
        questions.c.question_document_id,
        questions.c.question_number,
        questions.c.normalized_question_number,
        questions.c.occurrence_index,
        questions.c.question_text,
        questions.c.context_text,
        questions.c.search_text,
        questions.c.marks,
        questions.c.page_start,
        questions.c.page_end,
        questions.c.has_code,
        questions.c.has_visual,
        questions.c.visual_page_numbers,
        questions.c.review_status,
        questions.c.embedding_status,
        questions.c.content_hash.label("question_content_hash"),

        topics.c.pmt_topic_number.label("topic_number"),
        topics.c.pmt_topic_name.label("topic_name"),
        topics.c.pmt_subtopic_code.label("subtopic_code"),
        topics.c.pmt_subtopic_name.label("subtopic_name"),
        topics.c.paper_code,
        topics.c.programming_language,

        question_ms_links.c.id.label("link_id"),
        question_ms_links.c.match_method,
        question_ms_links.c.match_confidence,

        mark_scheme_entries.c.id.label("mark_scheme_id"),
        mark_scheme_entries.c.mark_scheme_uid,
        mark_scheme_entries.c.maximum_marks,
        mark_scheme_entries.c.assessment_objectives,
    )
    .select_from(
        questions
        .join(
            topics,
            questions.c.topic_id == topics.c.id,
        )
        .join(
            question_ms_links,
            question_ms_links.c.question_id == questions.c.id,
        )
        .join(
            mark_scheme_entries,
            mark_scheme_entries.c.id
            == question_ms_links.c.mark_scheme_entry_id,
        )
    )
    .where(
        questions.c.record_type == "scored_item",
        questions.c.review_status.in_(
            ["human_approved", "human_corrected"]
        ),
        questions.c.retrieval_enabled.is_(True),
        questions.c.is_active.is_(True),
        questions.c.is_legacy.is_(False),
        questions.c.embedding_status.in_(
            ["ready_for_indexing", "indexed"]
        ),
        mark_scheme_entries.c.is_active.is_(True),
    )
    .order_by(
        topics.c.pmt_topic_number,
        topics.c.pmt_subtopic_code,
        questions.c.sequence_index,
    )
)


with engine.connect() as connection:
    eligible_df = pd.read_sql(
        eligible_query,
        connection,
    )


# Normalize UUIDs once so every later DataFrame merge uses
# the same hashable/string representation.
eligible_df["question_id"] = (
    eligible_df["question_id"]
    .astype(str)
)

print(f"Eligible scored questions: {len(eligible_df)}")

if len(eligible_df) != EXPECTED_CURRENT_ELIGIBLE_COUNT:
    print(
        "Warning: the current count differs from the "
        f"post-Notebook-03 expectation of "
        f"{EXPECTED_CURRENT_ELIGIBLE_COUNT}. "
        "This is acceptable only if the reviewed dataset changed."
    )

if eligible_df.empty:
    raise RuntimeError(
        "No retrieval-safe scored questions were found."
    )


eligibility_summary_df = (
    eligible_df.groupby(
        [
            "topic_number",
            "topic_name",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="question_count")
    .sort_values("topic_number")
)

display(eligibility_summary_df)


Eligible scored questions: 820


,topic_number,topic_name,question_count
0,1,Fundamentals of Algorithms,80
1,2,Programming,410
2,3,Fundamentals of Data Representation,134
3,4,Computer Systems,96
4,5,Fundamentals of Computer Networks,44
5,6,Cyber Security,28
6,7,Relational Databases and Structured Query Lang...,22
7,8,"Ethical, Legal and Environmental Impacts of Di...",6


## 6. Load MiniLM

`all-MiniLM-L6-v2` produces 384-dimensional vectors.

The embedding text is constructed so the topic, subtopic, and actual
question are preserved before optional context. This is important because
MiniLM truncates long inputs.


In [55]:
model_started = time.perf_counter()

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE,
    cache_folder=str(MODEL_CACHE_DIR),
)

VECTOR_SIZE = int(
    model.get_sentence_embedding_dimension()
)

MAX_SEQUENCE_LENGTH = int(
    model.max_seq_length
)

if VECTOR_SIZE != EXPECTED_VECTOR_SIZE:
    raise RuntimeError(
        f"Expected {EXPECTED_VECTOR_SIZE} dimensions, "
        f"but model returned {VECTOR_SIZE}."
    )

print(f"Model loaded in:      {time.perf_counter() - model_started:.2f}s")
print(f"Vector size:          {VECTOR_SIZE}")
print(f"Max sequence length:  {MAX_SEQUENCE_LENGTH}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded in:      6.84s
Vector size:          384
Max sequence length:  256


C:\Users\hp\AppData\Local\Temp\ipykernel_10960\274292820.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


## 7. Build deterministic, token-aware embedding text


In [56]:
def utc_now() -> datetime:
    return datetime.now(timezone.utc)


def strict_json_value(value: Any) -> Any:
    if value is None:
        return None

    if isinstance(value, bool):
        return value

    if isinstance(value, int):
        return value

    if isinstance(value, float):
        return value if math.isfinite(value) else None

    if isinstance(value, str):
        return value

    if isinstance(value, dict):
        return {
            str(key): strict_json_value(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [
            strict_json_value(item)
            for item in value
        ]

    if isinstance(value, (uuid.UUID, Path)):
        return str(value)

    if isinstance(value, datetime):
        return value.isoformat()

    if hasattr(value, "item"):
        try:
            scalar = value.item()
            if scalar is not value:
                return strict_json_value(scalar)
        except (TypeError, ValueError):
            pass

    try:
        missing = pd.isna(value)

        if isinstance(missing, bool) and missing:
            return None

        if hasattr(missing, "item") and bool(missing.item()):
            return None

    except (TypeError, ValueError):
        pass

    return str(value)


def sha256_json(payload: dict[str, Any]) -> str:
    encoded = json.dumps(
        strict_json_value(payload),
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

    return hashlib.sha256(encoded).hexdigest()


def tokenize_without_special_tokens(text: str) -> list[int]:
    """
    Build the complete token-id list without asking Transformers to
    validate it against the model limit. Manual truncation happens below.
    """
    tokens = model.tokenizer.tokenize(text)

    return model.tokenizer.convert_tokens_to_ids(
        tokens
    )


def build_embedding_text(
    row: pd.Series,
) -> tuple[str, int, bool]:
    topic_header = (
        f"GCSE Computer Science. "
        f"Topic: {row['topic_name']}. "
        f"Subtopic: {row['subtopic_name']}."
    )

    question_section = (
        f"Question: "
        f"{str(row['question_text']).strip()}"
    )

    base_text = (
        f"{topic_header}\n"
        f"{question_section}"
    ).strip()

    context_text = str(
        row.get("context_text") or ""
    ).strip()

    base_tokens = tokenize_without_special_tokens(
        base_text
    )

    special_token_allowance = 2
    usable_limit = max(
        1,
        MAX_SEQUENCE_LENGTH - special_token_allowance,
    )

    if len(base_tokens) >= usable_limit:
        final_tokens = base_tokens[:usable_limit]
        final_text = model.tokenizer.decode(
            final_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )

        return (
            final_text.strip(),
            len(final_tokens),
            True,
        )

    if not context_text:
        return (
            base_text,
            len(base_tokens),
            False,
        )

    context_prefix = "\nContext: "
    prefix_tokens = tokenize_without_special_tokens(
        context_prefix
    )

    remaining = (
        usable_limit
        - len(base_tokens)
        - len(prefix_tokens)
    )

    if remaining <= 0:
        return (
            base_text,
            len(base_tokens),
            True,
        )

    context_tokens = tokenize_without_special_tokens(
        context_text
    )

    used_context_tokens = context_tokens[:remaining]

    decoded_context = model.tokenizer.decode(
        used_context_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    final_text = (
        f"{base_text}"
        f"{context_prefix}"
        f"{decoded_context}"
    ).strip()

    truncated = (
        len(used_context_tokens)
        < len(context_tokens)
    )

    final_token_count = (
        len(base_tokens)
        + len(prefix_tokens)
        + len(used_context_tokens)
    )

    return (
        final_text,
        final_token_count,
        truncated,
    )


embedding_text_records = []

for _, row in eligible_df.iterrows():
    (
        embedding_text,
        token_count,
        was_truncated,
    ) = build_embedding_text(row)

    embedding_text_hash = sha256_json(
        {
            "model": MODEL_NAME,
            "text": embedding_text,
        }
    )

    embedding_text_records.append(
        {
            "question_id": str(row["question_id"]),
            "embedding_text": embedding_text,
            "embedding_token_count": token_count,
            "embedding_was_truncated": was_truncated,
            "embedding_text_hash": embedding_text_hash,
        }
    )


embedding_text_df = pd.DataFrame(
    embedding_text_records
)

eligible_df = eligible_df.merge(
    embedding_text_df,
    on="question_id",
    how="left",
    validate="one_to_one",
)


truncation_summary_df = pd.DataFrame(
    [
        {
            "total_questions": len(eligible_df),
            "truncated_questions": int(
                eligible_df[
                    "embedding_was_truncated"
                ].sum()
            ),
            "max_token_count": (
                int(
                    eligible_df[
                        "embedding_token_count"
                    ].max()
                )
                if (
                    not eligible_df.empty
                    and eligible_df[
                        "embedding_token_count"
                    ].notna().any()
                )
                else 0
            ),
            "median_token_count": (
                float(
                    eligible_df[
                        "embedding_token_count"
                    ].median()
                )
                if (
                    not eligible_df.empty
                    and eligible_df[
                        "embedding_token_count"
                    ].notna().any()
                )
                else 0.0
            ),
        }
    ]
)

display(truncation_summary_df)

display(
    eligible_df[
        [
            "topic_name",
            "subtopic_name",
            "question_number",
            "embedding_token_count",
            "embedding_was_truncated",
            "embedding_text",
        ]
    ].head(10)
)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (300 > 256). Running this sequence through the model will result in indexing errors


,total_questions,truncated_questions,max_token_count,median_token_count
0,820,100,254,134.0


,topic_name,subtopic_name,question_number,embedding_token_count,embedding_was_truncated,embedding_text
0,Fundamentals of Algorithms,Representing Algorithms,01.1,27,False,GCSE Computer Science. Topic: Fundamentals of ...
1,Fundamentals of Algorithms,Representing Algorithms,01.2,110,False,GCSE Computer Science. Topic: Fundamentals of ...
2,Fundamentals of Algorithms,Representing Algorithms,02,182,False,GCSE Computer Science. Topic: Fundamentals of ...
3,Fundamentals of Algorithms,Representing Algorithms,03,148,False,GCSE Computer Science. Topic: Fundamentals of ...
4,Fundamentals of Algorithms,Representing Algorithms,04.1,156,False,GCSE Computer Science. Topic: Fundamentals of ...
5,Fundamentals of Algorithms,Representing Algorithms,04.2,154,False,GCSE Computer Science. Topic: Fundamentals of ...
6,Fundamentals of Algorithms,Representing Algorithms,04.3,212,False,GCSE Computer Science. Topic: Fundamentals of ...
7,Fundamentals of Algorithms,Representing Algorithms,04.4,229,False,GCSE Computer Science. Topic: Fundamentals of ...
8,Fundamentals of Algorithms,Representing Algorithms,05.1,137,False,GCSE Computer Science. Topic: Fundamentals of ...
9,Fundamentals of Algorithms,Representing Algorithms,05.2,49,False,GCSE Computer Science. Topic: Fundamentals of ...


## 8. Select the pilot or complete dataset


In [57]:
def select_distributed_pilot(
    frame: pd.DataFrame,
    target_size: int,
) -> pd.DataFrame:
    selected_indexes: list[int] = []

    grouped = frame.groupby(
        "topic_number",
        sort=True,
    )

    while len(selected_indexes) < target_size:
        added_this_round = False

        for _, group in grouped:
            next_position = sum(
                frame.loc[index, "topic_number"]
                == group.iloc[0]["topic_number"]
                for index in selected_indexes
            )

            if next_position < len(group):
                selected_indexes.append(
                    int(group.index[next_position])
                )
                added_this_round = True

            if len(selected_indexes) >= target_size:
                break

        if not added_this_round:
            break

    return (
        frame.loc[selected_indexes]
        .sort_values(
            [
                "topic_number",
                "subtopic_code",
                "question_number",
            ]
        )
        .reset_index(drop=True)
    )


selected_df = (
    select_distributed_pilot(
        eligible_df,
        min(PILOT_SIZE, len(eligible_df)),
    )
    if RUN_MODE == "pilot"
    else eligible_df.copy()
)


print(f"Selected questions: {len(selected_df)}")
print(f"Run mode:           {RUN_MODE}")

display(
    selected_df[
        [
            "question_id",
            "topic_name",
            "subtopic_name",
            "question_number",
            "marks",
            "embedding_status",
        ]
    ].head(40)
)


Selected questions: 820
Run mode:           full


,question_id,topic_name,subtopic_name,question_number,marks,embedding_status
0,d6b585dd-e3d3-497f-b9b2-75ea589fa120,Fundamentals of Algorithms,Representing Algorithms,01.1,2,ready_for_indexing
1,d1a19dd3-1edb-46e8-93d4-026a3665c499,Fundamentals of Algorithms,Representing Algorithms,01.2,3,ready_for_indexing
2,0332d76f-c80e-4b93-95e2-7706bbd8dab6,Fundamentals of Algorithms,Representing Algorithms,02,7,ready_for_indexing
3,e7acf7bb-291e-4aa0-879a-6cda0b3162aa,Fundamentals of Algorithms,Representing Algorithms,03,8,ready_for_indexing
4,6ea35f5a-c317-46d2-b243-645ce9625f8c,Fundamentals of Algorithms,Representing Algorithms,04.1,3,ready_for_indexing
5,b03ac3af-d6bc-4805-9b3b-1e405867cd28,Fundamentals of Algorithms,Representing Algorithms,04.2,3,ready_for_indexing
6,6eab1253-bbd5-4382-b222-a156fd834bfe,Fundamentals of Algorithms,Representing Algorithms,04.3,3,ready_for_indexing
7,f17d176b-8946-4018-b692-c3cdc661b129,Fundamentals of Algorithms,Representing Algorithms,04.4,5,ready_for_indexing
8,9db09e6b-17f8-458d-854c-1806b2267666,Fundamentals of Algorithms,Representing Algorithms,05.1,4,ready_for_indexing
9,b266b83d-13a5-4fcf-af8e-d06bb670883e,Fundamentals of Algorithms,Representing Algorithms,05.2,1,ready_for_indexing


## 9. Determine which records require indexing

A question is embedded again only when:

- it has never been indexed;
- its embedding text changed;
- the model or collection changed;
- the previous indexing attempt failed; or
- `FORCE_REINDEX=True`.


In [58]:
with engine.connect() as connection:
    existing_index_df = pd.read_sql(
        select(question_vector_index)
        .where(
            question_vector_index.c.collection_name
            == COLLECTION_NAME
        ),
        connection,
    )


if existing_index_df.empty:
    existing_lookup = {}
else:
    existing_lookup = {
        str(row.question_id): row
        for row in existing_index_df.itertuples()
    }


indexing_decisions = []

for _, row in selected_df.iterrows():
    question_id = str(row["question_id"])
    existing = existing_lookup.get(question_id)

    reasons: list[str] = []

    if FORCE_REINDEX:
        reasons.append("force_reindex")

    if existing is None:
        reasons.append("not_previously_indexed")

    else:
        if (
            existing.embedding_text_hash
            != row["embedding_text_hash"]
        ):
            reasons.append("embedding_text_changed")

        if existing.embedding_model != MODEL_NAME:
            reasons.append("embedding_model_changed")

        if (
            int(existing.embedding_dimension)
            != VECTOR_SIZE
        ):
            reasons.append("embedding_dimension_changed")

        if existing.vector_status != "indexed":
            reasons.append(
                f"previous_status={existing.vector_status}"
            )

    needs_indexing = bool(reasons)

    indexing_decisions.append(
        {
            "question_id": question_id,
            "needs_indexing": needs_indexing,
            "indexing_reasons": reasons,
        }
    )


decision_df = pd.DataFrame(
    indexing_decisions
)

selected_df = selected_df.merge(
    decision_df,
    on="question_id",
    how="left",
    validate="one_to_one",
)

pending_df = selected_df[
    selected_df["needs_indexing"]
].copy()

skipped_df = selected_df[
    ~selected_df["needs_indexing"]
].copy()


print(f"Requires embedding/indexing: {len(pending_df)}")
print(f"Already current:             {len(skipped_df)}")

display(
    selected_df[
        [
            "question_id",
            "question_number",
            "needs_indexing",
            "indexing_reasons",
        ]
    ].head(40)
)


Requires embedding/indexing: 820
Already current:             0


,question_id,question_number,needs_indexing,indexing_reasons
0,d6b585dd-e3d3-497f-b9b2-75ea589fa120,01.1,True,[not_previously_indexed]
1,d1a19dd3-1edb-46e8-93d4-026a3665c499,01.2,True,[not_previously_indexed]
2,0332d76f-c80e-4b93-95e2-7706bbd8dab6,02,True,[not_previously_indexed]
3,e7acf7bb-291e-4aa0-879a-6cda0b3162aa,03,True,[not_previously_indexed]
4,6ea35f5a-c317-46d2-b243-645ce9625f8c,04.1,True,[not_previously_indexed]
5,b03ac3af-d6bc-4805-9b3b-1e405867cd28,04.2,True,[not_previously_indexed]
6,6eab1253-bbd5-4382-b222-a156fd834bfe,04.3,True,[not_previously_indexed]
7,f17d176b-8946-4018-b692-c3cdc661b129,04.4,True,[not_previously_indexed]
8,9db09e6b-17f8-458d-854c-1806b2267666,05.1,True,[not_previously_indexed]
9,b266b83d-13a5-4fcf-af8e-d06bb670883e,05.2,True,[not_previously_indexed]


## 10. Generate MiniLM embeddings


In [59]:
embedding_started = time.perf_counter()

if pending_df.empty:
    pending_embeddings = np.empty(
        (0, VECTOR_SIZE),
        dtype=np.float32,
    )

    print(
        "No embeddings were required; all selected "
        "records are already current."
    )

else:
    pending_embeddings = model.encode(
        pending_df["embedding_text"].tolist(),
        batch_size=EMBEDDING_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)


if pending_embeddings.shape != (
    len(pending_df),
    VECTOR_SIZE,
):
    raise RuntimeError(
        "Unexpected embedding matrix shape: "
        f"{pending_embeddings.shape}"
    )


if len(pending_embeddings):
    vector_norms = np.linalg.norm(
        pending_embeddings,
        axis=1,
    )

    if not np.allclose(
        vector_norms,
        1.0,
        atol=1e-4,
    ):
        raise RuntimeError(
            "Embeddings were expected to be L2-normalized."
        )

else:
    vector_norms = np.array([])


print(
    f"Embedding time: "
    f"{time.perf_counter() - embedding_started:.2f}s"
)

print(
    f"Generated embeddings: {len(pending_embeddings)}"
)

if len(vector_norms):
    print(
        f"Vector norm range: "
        f"{vector_norms.min():.6f} – "
        f"{vector_norms.max():.6f}"
    )


Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Embedding time: 36.36s
Generated embeddings: 820
Vector norm range: 1.000000 – 1.000000


## 11. Connect to Qdrant


In [60]:
qdrant_client_kwargs = {
    "url": QDRANT_URL,
    "prefer_grpc": PREFER_GRPC,
    "timeout": QDRANT_TIMEOUT_SECONDS,
}

# Do not send an empty API key to local Docker Qdrant.
if QDRANT_API_KEY is not None:
    qdrant_client_kwargs["api_key"] = QDRANT_API_KEY

qdrant_client = QdrantClient(
    **qdrant_client_kwargs
)

collections_response = (
    qdrant_client.get_collections()
)

available_collections = sorted(
    collection.name
    for collection
    in collections_response.collections
)

print("Qdrant connection successful.")
print(
    f"Existing collections: "
    f"{len(available_collections)}"
)


Qdrant connection successful.
Existing collections: 1


## 12. Create or validate the Qdrant collection


In [61]:
def collection_exists(
    client: QdrantClient,
    collection_name: str,
) -> bool:
    if hasattr(client, "collection_exists"):
        return bool(
            client.collection_exists(
                collection_name
            )
        )

    try:
        client.get_collection(
            collection_name
        )
        return True

    except UnexpectedResponse as error:
        if getattr(error, "status_code", None) == 404:
            return False
        raise


def get_vector_configuration(
    collection_info: Any,
) -> tuple[int | None, str | None]:
    vectors_config = (
        collection_info
        .config
        .params
        .vectors
    )

    if isinstance(vectors_config, dict):
        return None, "named_vectors"

    size = getattr(
        vectors_config,
        "size",
        None,
    )

    distance = getattr(
        vectors_config,
        "distance",
        None,
    )

    distance_value = (
        str(distance.value)
        if hasattr(distance, "value")
        else str(distance)
    )

    return (
        int(size) if size is not None else None,
        distance_value.lower(),
    )


exists = collection_exists(
    qdrant_client,
    COLLECTION_NAME,
)


if exists and RECREATE_COLLECTION:
    if not COMMIT_TO_QDRANT:
        print(
            "Collection recreation requested, but "
            "COMMIT_TO_QDRANT=False. No changes made."
        )

    else:
        qdrant_client.delete_collection(
            collection_name=COLLECTION_NAME,
        )

        exists = False

        print(
            f"Deleted existing collection: "
            f"{COLLECTION_NAME}"
        )


if not exists:
    if not COMMIT_TO_QDRANT:
        print(
            "Collection does not exist. It will be created "
            "during the final committed run."
        )

    else:
        qdrant_client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=models.VectorParams(
                size=VECTOR_SIZE,
                distance=models.Distance.COSINE,
            ),
        )

        exists = True

        print(
            f"Created collection: {COLLECTION_NAME}"
        )


if exists:
    collection_info = (
        qdrant_client.get_collection(
            COLLECTION_NAME
        )
    )

    (
        existing_vector_size,
        existing_distance,
    ) = get_vector_configuration(
        collection_info
    )

    print(
        f"Existing vector size: {existing_vector_size}"
    )

    print(
        f"Existing distance:    {existing_distance}"
    )

    if existing_vector_size != VECTOR_SIZE:
        raise RuntimeError(
            "Qdrant vector-size mismatch. "
            "Set RECREATE_COLLECTION=True only if replacing "
            "the existing collection is intended."
        )

    if (
        existing_distance
        and "cosine" not in existing_distance
    ):
        raise RuntimeError(
            "The existing Qdrant collection does not use "
            "cosine distance."
        )


Created collection: aqa_gcse_computer_science_8525_questions
Existing vector size: 384
Existing distance:    cosine


## 13. Create useful Qdrant payload indexes

These fields support efficient filtered semantic retrieval.


In [62]:
PAYLOAD_INDEXES = {
    "question_id": models.PayloadSchemaType.KEYWORD,
    "question_uid": models.PayloadSchemaType.KEYWORD,
    "pair_key": models.PayloadSchemaType.KEYWORD,
    "topic_number": models.PayloadSchemaType.INTEGER,
    "topic_name": models.PayloadSchemaType.KEYWORD,
    "subtopic_code": models.PayloadSchemaType.KEYWORD,
    "paper_code": models.PayloadSchemaType.KEYWORD,
    "programming_language": models.PayloadSchemaType.KEYWORD,
    "marks": models.PayloadSchemaType.INTEGER,
    "has_code": models.PayloadSchemaType.BOOL,
    "has_visual": models.PayloadSchemaType.BOOL,
    "review_status": models.PayloadSchemaType.KEYWORD,
}


payload_index_results = []

if not exists:
    print(
        "Payload indexes will be created after the "
        "collection exists."
    )

elif not COMMIT_TO_QDRANT:
    print(
        "Payload-index creation skipped because "
        "COMMIT_TO_QDRANT=False."
    )

else:
    for (
        field_name,
        field_schema,
    ) in PAYLOAD_INDEXES.items():
        try:
            qdrant_client.create_payload_index(
                collection_name=COLLECTION_NAME,
                field_name=field_name,
                field_schema=field_schema,
                wait=True,
            )

            payload_index_results.append(
                {
                    "field": field_name,
                    "status": "created_or_already_available",
                    "error": None,
                }
            )

        except Exception as error:
            error_text = str(error)

            if "already exists" in error_text.lower():
                status = "already_exists"
                error_text = None
            else:
                status = "warning"

            payload_index_results.append(
                {
                    "field": field_name,
                    "status": status,
                    "error": error_text,
                }
            )


payload_index_results_df = pd.DataFrame(
    payload_index_results
)

display(payload_index_results_df)


,field,status,error
0,question_id,created_or_already_available,None
1,question_uid,created_or_already_available,None
2,pair_key,created_or_already_available,None
3,topic_number,created_or_already_available,None
4,topic_name,created_or_already_available,None
5,subtopic_code,created_or_already_available,None
6,paper_code,created_or_already_available,None
7,programming_language,created_or_already_available,None
8,marks,created_or_already_available,None
9,has_code,created_or_already_available,None


## 14. Build Qdrant payloads and deterministic point IDs


In [63]:
def build_qdrant_payload(
    row: pd.Series,
) -> dict[str, Any]:
    payload = {
        "question_id": str(row["question_id"]),
        "question_uid": row["question_uid"],
        "pair_key": row["pair_key"],

        "topic_id": str(row["topic_id"]),
        "topic_number": int(row["topic_number"]),
        "topic_name": row["topic_name"],
        "subtopic_code": str(row["subtopic_code"]),
        "subtopic_name": row["subtopic_name"],
        "paper_code": row["paper_code"],
        "programming_language": (
            row["programming_language"]
            if pd.notna(row["programming_language"])
            else None
        ),

        "question_number": row["question_number"],
        "normalized_question_number": (
            row["normalized_question_number"]
        ),
        "question_text": row["question_text"],
        "context_text": row["context_text"] or "",
        "marks": int(row["marks"]),

        "page_start": int(row["page_start"]),
        "page_end": int(row["page_end"]),
        "has_code": bool(row["has_code"]),
        "has_visual": bool(row["has_visual"]),
        "visual_page_numbers": (
            row["visual_page_numbers"]
            if isinstance(
                row["visual_page_numbers"],
                list,
            )
            else []
        ),

        "review_status": row["review_status"],
        "question_content_hash": (
            row["question_content_hash"]
        ),

        "link_id": str(row["link_id"]),
        "match_method": row["match_method"],
        "match_confidence": float(
            row["match_confidence"]
        ),

        "mark_scheme_id": str(
            row["mark_scheme_id"]
        ),
        "mark_scheme_uid": row["mark_scheme_uid"],
        "maximum_marks": int(
            row["maximum_marks"]
        ),
        "assessment_objectives": (
            row["assessment_objectives"]
            if isinstance(
                row["assessment_objectives"],
                list,
            )
            else []
        ),

        "embedding_model": MODEL_NAME,
        "embedding_dimension": VECTOR_SIZE,
        "embedding_text_hash": (
            row["embedding_text_hash"]
        ),
        "embedding_token_count": int(
            row["embedding_token_count"]
        ),
        "embedding_was_truncated": bool(
            row["embedding_was_truncated"]
        ),
        "indexer_version": INDEXER_VERSION,
    }

    return strict_json_value(payload)


payload_records = []

for _, row in pending_df.iterrows():
    payload = build_qdrant_payload(row)

    payload_records.append(
        {
            "question_id": str(row["question_id"]),
            "qdrant_point_id": str(row["question_id"]),
            "payload": payload,
            "payload_hash": sha256_json(payload),
        }
    )


payload_df = pd.DataFrame(
    payload_records
)

print(f"Payload records: {len(payload_df)}")

display(
    payload_df.head(5)
)


Payload records: 820


,question_id,qdrant_point_id,payload,payload_hash
0,d6b585dd-e3d3-497f-b9b2-75ea589fa120,d6b585dd-e3d3-497f-b9b2-75ea589fa120,{'question_id': 'd6b585dd-e3d3-497f-b9b2-75ea5...,3c23c12b0a5baeece9abe08f852a6eb76a7b9cdecdf158...
1,d1a19dd3-1edb-46e8-93d4-026a3665c499,d1a19dd3-1edb-46e8-93d4-026a3665c499,{'question_id': 'd1a19dd3-1edb-46e8-93d4-026a3...,3ef53b0a5ba5715391a5c19b8f307bce4cba768a5b3fa9...
2,0332d76f-c80e-4b93-95e2-7706bbd8dab6,0332d76f-c80e-4b93-95e2-7706bbd8dab6,{'question_id': '0332d76f-c80e-4b93-95e2-7706b...,aaa8b52d2f7a7d6e6dd35b04827b185dc265795a43c411...
3,e7acf7bb-291e-4aa0-879a-6cda0b3162aa,e7acf7bb-291e-4aa0-879a-6cda0b3162aa,{'question_id': 'e7acf7bb-291e-4aa0-879a-6cda0...,b294d22134dba1a7e10057f518e1bf9b4e5bce54be1297...
4,6ea35f5a-c317-46d2-b243-645ce9625f8c,6ea35f5a-c317-46d2-b243-645ce9625f8c,{'question_id': '6ea35f5a-c317-46d2-b243-645ce...,aafe89b643c0c6103fcd2a3757a98f9f3f623cf31b288d...


## 15. Optional stale-point cleanup

This runs only during a committed **full** run.

A stale point is one that was indexed previously but is no longer retrieval-eligible.


In [64]:
stale_point_ids: list[str] = []

if (
    RUN_MODE == "full"
    and DELETE_STALE_POINTS
):
    current_eligible_ids = set(
        eligible_df["question_id"].astype(str)
    )

    if existing_index_df.empty:
        stale_index_rows_df = (
            existing_index_df.copy()
        )

    else:
        stale_index_rows_df = (
            existing_index_df[
                (
                    existing_index_df["collection_name"]
                    == COLLECTION_NAME
                )
                & (
                    existing_index_df["vector_status"]
                    == "indexed"
                )
                & (
                    ~existing_index_df[
                        "question_id"
                    ]
                    .astype(str)
                    .isin(current_eligible_ids)
                )
            ].copy()
        )

    stale_point_ids = (
        stale_index_rows_df[
            "qdrant_point_id"
        ]
        .astype(str)
        .tolist()
        if not stale_index_rows_df.empty
        else []
    )

else:
    stale_index_rows_df = pd.DataFrame()


print(f"Stale points identified: {len(stale_point_ids)}")


Stale points identified: 0


## 16. Upsert MiniLM vectors into Qdrant


In [65]:
def batches(
    values: list[Any],
    batch_size: int,
) -> Iterable[list[Any]]:
    for start in range(
        0,
        len(values),
        batch_size,
    ):
        yield values[
            start : start + batch_size
        ]


indexing_run_id: uuid.UUID | None = None

successful_question_ids: list[str] = []
failed_records: list[dict[str, Any]] = []
stale_deleted_count = 0


run_configuration = {
    "run_mode": RUN_MODE,
    "commit_to_qdrant": COMMIT_TO_QDRANT,
    "force_reindex": FORCE_REINDEX,
    "delete_stale_points": DELETE_STALE_POINTS,
    "embedding_batch_size": EMBEDDING_BATCH_SIZE,
    "qdrant_upsert_batch_size": QDRANT_UPSERT_BATCH_SIZE,
    "device": DEVICE,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "qdrant_url": QDRANT_URL,
}


if not COMMIT_TO_QDRANT:
    print(
        "Dry run complete. No Qdrant points or PostgreSQL "
        "index statuses were changed."
    )

else:
    if not collection_exists(
        qdrant_client,
        COLLECTION_NAME,
    ):
        raise RuntimeError(
            "The Qdrant collection does not exist."
        )

    indexing_run_id = uuid.uuid4()

    with Session(engine) as session:
        session.execute(
            insert(embedding_runs).values(
                id=indexing_run_id,
                indexer_version=INDEXER_VERSION,
                embedding_model=MODEL_NAME,
                embedding_dimension=VECTOR_SIZE,
                collection_name=COLLECTION_NAME,
                run_mode=RUN_MODE,
                committed=True,
                status="running",
                source_count=len(selected_df),
                embedded_count=len(pending_df),
                indexed_count=0,
                skipped_count=len(skipped_df),
                failed_count=0,
                deleted_stale_count=0,
                configuration=run_configuration,
                started_at=utc_now(),
            )
        )

        session.commit()


    if stale_point_ids:
        try:
            qdrant_client.delete(
                collection_name=COLLECTION_NAME,
                points_selector=models.PointIdsList(
                    points=stale_point_ids
                ),
                wait=True,
            )

            stale_deleted_count = len(
                stale_point_ids
            )

            with Session(engine) as session:
                session.execute(
                    update(question_vector_index)
                    .where(
                        question_vector_index.c.collection_name
                        == COLLECTION_NAME,
                        question_vector_index.c.qdrant_point_id.in_(
                            stale_point_ids
                        ),
                    )
                    .values(
                        vector_status="deleted_stale",
                        last_verified_at=utc_now(),
                        updated_at=utc_now(),
                    )
                )

                session.commit()

        except Exception as error:
            print(
                "Warning: stale-point deletion failed: "
                f"{type(error).__name__}: {error}"
            )


    point_records = []

    for position, (_, row) in enumerate(
        pending_df.iterrows()
    ):
        payload_record = payload_df.iloc[position]

        point_records.append(
            models.PointStruct(
                id=payload_record[
                    "qdrant_point_id"
                ],
                vector=pending_embeddings[
                    position
                ].tolist(),
                payload=payload_record[
                    "payload"
                ],
            )
        )


    indexed_at = utc_now()

    for point_batch in batches(
        point_records,
        QDRANT_UPSERT_BATCH_SIZE,
    ):
        batch_ids = [
            str(point.id)
            for point in point_batch
        ]

        try:
            qdrant_client.upsert(
                collection_name=COLLECTION_NAME,
                points=point_batch,
                wait=True,
            )

            successful_question_ids.extend(
                batch_ids
            )

            batch_rows = pending_df[
                pending_df["question_id"]
                .astype(str)
                .isin(batch_ids)
            ]

            with Session(engine) as session:
                for _, row in batch_rows.iterrows():
                    payload_row = payload_df[
                        payload_df[
                            "question_id"
                        ]
                        == str(
                            row["question_id"]
                        )
                    ].iloc[0]

                    statement = (
                        pg_insert(
                            question_vector_index
                        )
                        .values(
                            id=uuid.uuid4(),
                            question_id=uuid.UUID(
                                str(row["question_id"])
                            ),
                            run_id=indexing_run_id,
                            qdrant_point_id=str(
                                row["question_id"]
                            ),
                            collection_name=(
                                COLLECTION_NAME
                            ),
                            embedding_model=MODEL_NAME,
                            embedding_dimension=(
                                VECTOR_SIZE
                            ),
                            embedding_text_hash=row[
                                "embedding_text_hash"
                            ],
                            payload_hash=payload_row[
                                "payload_hash"
                            ],
                            vector_status="indexed",
                            indexed_at=indexed_at,
                            last_verified_at=indexed_at,
                            error_message=None,
                            created_at=indexed_at,
                            updated_at=indexed_at,
                        )
                        .on_conflict_do_update(
                            constraint=(
                                "uq_question_vector_index_"
                                "question_collection"
                            ),
                            set_={
                                "run_id": indexing_run_id,
                                "qdrant_point_id": str(
                                    row["question_id"]
                                ),
                                "embedding_model": MODEL_NAME,
                                "embedding_dimension": (
                                    VECTOR_SIZE
                                ),
                                "embedding_text_hash": row[
                                    "embedding_text_hash"
                                ],
                                "payload_hash": payload_row[
                                    "payload_hash"
                                ],
                                "vector_status": "indexed",
                                "indexed_at": indexed_at,
                                "last_verified_at": (
                                    indexed_at
                                ),
                                "error_message": None,
                                "updated_at": indexed_at,
                            },
                        )
                    )

                    session.execute(statement)

                session.execute(
                    update(questions)
                    .where(
                        questions.c.id.in_(
                            [
                                uuid.UUID(
                                    question_id
                                )
                                for question_id
                                in batch_ids
                            ]
                        )
                    )
                    .values(
                        embedding_status="indexed",
                        updated_at=indexed_at,
                    )
                )

                session.commit()

            print(
                f"Indexed batch: {len(batch_ids)}"
            )

        except Exception as error:
            error_text = (
                f"{type(error).__name__}: {error}"
            )

            for question_id in batch_ids:
                failed_records.append(
                    {
                        "question_id": question_id,
                        "error": error_text,
                    }
                )

            with Session(engine) as session:
                session.execute(
                    update(questions)
                    .where(
                        questions.c.id.in_(
                            [
                                uuid.UUID(
                                    question_id
                                )
                                for question_id
                                in batch_ids
                            ]
                        )
                    )
                    .values(
                        embedding_status=(
                            "indexing_failed"
                        ),
                        updated_at=utc_now(),
                    )
                )

                session.commit()

            print(
                f"Failed batch: {error_text}"
            )


    with Session(engine) as session:
        session.execute(
            update(embedding_runs)
            .where(
                embedding_runs.c.id
                == indexing_run_id
            )
            .values(
                status=(
                    "completed"
                    if not failed_records
                    else "completed_with_failures"
                ),
                indexed_count=len(
                    successful_question_ids
                ),
                failed_count=len(
                    failed_records
                ),
                deleted_stale_count=(
                    stale_deleted_count
                ),
                completed_at=utc_now(),
            )
        )

        session.commit()


    print("\nIndexing summary:")
    print(
        f"Successfully indexed: "
        f"{len(successful_question_ids)}"
    )
    print(
        f"Failed:               "
        f"{len(failed_records)}"
    )
    print(
        f"Skipped unchanged:    "
        f"{len(skipped_df)}"
    )
    print(
        f"Deleted stale:        "
        f"{stale_deleted_count}"
    )


Indexed batch: 64
Indexed batch: 64
Indexed batch: 64
Indexed batch: 64
Indexed batch: 64
Indexed batch: 64
Indexed batch: 64
Indexed batch: 64
Indexed batch: 64
Indexed batch: 64
Indexed batch: 64
Indexed batch: 64
Indexed batch: 52

Indexing summary:
Successfully indexed: 820
Failed:               0
Skipped unchanged:    0
Deleted stale:        0


## 17. Qdrant collection and PostgreSQL index counts


In [66]:
if collection_exists(
    qdrant_client,
    COLLECTION_NAME,
):
    qdrant_point_count = int(
        qdrant_client.count(
            collection_name=COLLECTION_NAME,
            exact=True,
        ).count
    )

else:
    qdrant_point_count = 0


with engine.connect() as connection:
    postgres_indexed_count = int(
        connection.scalar(
            select(func.count())
            .select_from(
                question_vector_index
            )
            .where(
                question_vector_index.c.collection_name
                == COLLECTION_NAME,
                question_vector_index.c.vector_status
                == "indexed",
            )
        )
        or 0
    )

    indexed_question_count = int(
        connection.scalar(
            select(func.count())
            .select_from(questions)
            .where(
                questions.c.record_type
                == "scored_item",
                questions.c.retrieval_enabled
                .is_(True),
                questions.c.is_active.is_(True),
                questions.c.is_legacy.is_(False),
                questions.c.embedding_status
                == "indexed",
            )
        )
        or 0
    )

    ready_for_indexing_count = int(
        connection.scalar(
            select(func.count())
            .select_from(questions)
            .where(
                questions.c.record_type
                == "scored_item",
                questions.c.retrieval_enabled
                .is_(True),
                questions.c.is_active.is_(True),
                questions.c.is_legacy.is_(False),
                questions.c.embedding_status
                == "ready_for_indexing",
            )
        )
        or 0
    )


index_counts_df = pd.DataFrame(
    [
        {
            "metric": "eligible_postgres_questions",
            "value": len(eligible_df),
        },
        {
            "metric": "qdrant_point_count",
            "value": qdrant_point_count,
        },
        {
            "metric": "postgres_vector_index_count",
            "value": postgres_indexed_count,
        },
        {
            "metric": "questions_embedding_status_indexed",
            "value": indexed_question_count,
        },
        {
            "metric": "questions_ready_for_indexing",
            "value": ready_for_indexing_count,
        },
    ]
)

display(index_counts_df)


,metric,value
0,eligible_postgres_questions,820
1,qdrant_point_count,820
2,postgres_vector_index_count,820
3,questions_embedding_status_indexed,820
4,questions_ready_for_indexing,0


## 18. Semantic-search helper with optional metadata filters


In [67]:
def build_qdrant_filter(
    *,
    topic_number: int | None = None,
    subtopic_code: str | None = None,
    paper_code: str | None = None,
    programming_language: str | None = None,
    marks: int | None = None,
    has_code: bool | None = None,
    has_visual: bool | None = None,
) -> models.Filter | None:
    conditions: list[
        models.FieldCondition
    ] = []

    exact_values = {
        "topic_number": topic_number,
        "subtopic_code": subtopic_code,
        "paper_code": paper_code,
        "programming_language": (
            programming_language
        ),
        "marks": marks,
        "has_code": has_code,
        "has_visual": has_visual,
    }

    for key, value in exact_values.items():
        if value is None:
            continue

        conditions.append(
            models.FieldCondition(
                key=key,
                match=models.MatchValue(
                    value=value
                ),
            )
        )

    return (
        models.Filter(must=conditions)
        if conditions
        else None
    )


def semantic_search(
    query_text: str,
    *,
    limit: int = QUERY_LIMIT,
    score_threshold: float | None = None,
    topic_number: int | None = None,
    subtopic_code: str | None = None,
    paper_code: str | None = None,
    programming_language: str | None = None,
    marks: int | None = None,
    has_code: bool | None = None,
    has_visual: bool | None = None,
) -> pd.DataFrame:
    if not collection_exists(
        qdrant_client,
        COLLECTION_NAME,
    ):
        raise RuntimeError(
            "The Qdrant collection does not exist."
        )

    query_vector = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0].astype(np.float32)

    effective_score_threshold = (
        score_threshold
        if score_threshold is not None
        else QDRANT_SCORE_THRESHOLD
    )

    result = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector.tolist(),
        query_filter=build_qdrant_filter(
            topic_number=topic_number,
            subtopic_code=subtopic_code,
            paper_code=paper_code,
            programming_language=(
                programming_language
            ),
            marks=marks,
            has_code=has_code,
            has_visual=has_visual,
        ),
        limit=limit,
        score_threshold=effective_score_threshold,
        with_payload=True,
        with_vectors=False,
    )

    points = (
        result.points
        if hasattr(result, "points")
        else result
    )

    rows = []

    for rank, point in enumerate(
        points,
        start=1,
    ):
        payload = point.payload or {}

        rows.append(
            {
                "rank": rank,
                "score": float(point.score),
                "point_id": str(point.id),
                "question_id": payload.get(
                    "question_id"
                ),
                "topic": payload.get(
                    "topic_name"
                ),
                "subtopic": payload.get(
                    "subtopic_name"
                ),
                "question_number": payload.get(
                    "question_number"
                ),
                "marks": payload.get("marks"),
                "has_code": payload.get(
                    "has_code"
                ),
                "has_visual": payload.get(
                    "has_visual"
                ),
                "question_text": payload.get(
                    "question_text"
                ),
            }
        )

    return pd.DataFrame(rows)


## 19. Retrieve the full linked QP/MS bundle from PostgreSQL


In [68]:
def fetch_full_question_bundles(
    question_ids: list[str],
) -> pd.DataFrame:
    if not question_ids:
        return pd.DataFrame()

    question_uuids = [
        uuid.UUID(question_id)
        for question_id in question_ids
    ]

    query = (
        select(
            questions.c.id.label(
                "question_id"
            ),
            questions.c.question_uid,
            questions.c.question_number,
            questions.c.question_text,
            questions.c.context_text,
            questions.c.marks,
            questions.c.page_start,
            questions.c.page_end,
            topics.c.pmt_topic_name.label(
                "topic_name"
            ),
            topics.c.pmt_subtopic_name.label(
                "subtopic_name"
            ),
            mark_scheme_entries.c.id.label(
                "mark_scheme_id"
            ),
            mark_scheme_entries.c.maximum_marks,
            mark_scheme_entries.c.marking_guidance,
            mark_scheme_entries.c.marking_points,
            mark_scheme_entries.c.acceptable_answers,
            mark_scheme_entries.c.rejected_answers,
            mark_scheme_entries.c.additional_guidance,
            mark_scheme_entries.c.assessment_objectives,
        )
        .select_from(
            questions
            .join(
                topics,
                questions.c.topic_id
                == topics.c.id,
            )
            .join(
                question_ms_links,
                question_ms_links.c.question_id
                == questions.c.id,
            )
            .join(
                mark_scheme_entries,
                mark_scheme_entries.c.id
                == question_ms_links.c.mark_scheme_entry_id,
            )
        )
        .where(
            questions.c.id.in_(
                question_uuids
            )
        )
    )

    with engine.connect() as connection:
        bundle_df = pd.read_sql(
            query,
            connection,
        )

    requested_order = {
        question_id: rank
        for rank, question_id
        in enumerate(question_ids)
    }

    bundle_df["_rank"] = (
        bundle_df["question_id"]
        .astype(str)
        .map(requested_order)
    )

    return (
        bundle_df
        .sort_values("_rank")
        .drop(columns=["_rank"])
        .reset_index(drop=True)
    )


## 20. Semantic-search smoke tests


In [69]:
if qdrant_point_count == 0:
    print(
        "Search tests skipped because the collection "
        "contains no points."
    )

else:
    smoke_queries = [
        "How does binary search find an item?",
        "Explain how cyber security protects a network.",
        "Write an SQL query using a condition.",
    ]

    smoke_test_frames = []

    for query_text in smoke_queries:
        results_df = semantic_search(
            query_text,
            limit=5,
        )

        if not results_df.empty:
            results_df.insert(
                0,
                "query",
                query_text,
            )

            smoke_test_frames.append(
                results_df
            )


    semantic_smoke_df = (
        pd.concat(
            smoke_test_frames,
            ignore_index=True,
        )
        if smoke_test_frames
        else pd.DataFrame()
    )

    display(semantic_smoke_df)


,query,rank,score,point_id,question_id,topic,subtopic,question_number,marks,has_code,has_visual,question_text
0,How does binary search find an item?,1,0.573433,a1bc6778-1955-41cb-ab66-5c92747131c9,a1bc6778-1955-41cb-ab66-5c92747131c9,Fundamentals of Algorithms,Searching Algorithms,01.2,1,True,True,For a binary search algorithm to work correctl...
1,How does binary search find an item?,2,0.561719,ab675c37-0567-4d1a-a049-831625b0c892,ab675c37-0567-4d1a-a049-831625b0c892,Fundamentals of Algorithms,Searching Algorithms,03.3,1,False,False,State why a binary search cannot be used on th...
2,How does binary search find an item?,3,0.531949,1294b25d-0d6e-4f7f-aeec-5fe2a3d3896a,1294b25d-0d6e-4f7f-aeec-5fe2a3d3896a,Fundamentals of Algorithms,Searching Algorithms,01.1,3,True,False,State the comparisons that would be made if th...
3,How does binary search find an item?,4,0.466525,6b79f17d-5691-433f-acf3-542911b68485,6b79f17d-5691-433f-acf3-542911b68485,Fundamentals of Algorithms,Searching Algorithms,04,3,False,False,Explain how the linear search algorithm works.
4,How does binary search find an item?,5,0.458447,98e83707-ac9e-4a16-aec0-aa0f3b3be63d,98e83707-ac9e-4a16-aec0-aa0f3b3be63d,Fundamentals of Algorithms,Searching Algorithms,02,3,False,True,Describe how the linear search algorithm works.
5,Explain how cyber security protects a network.,1,0.656005,1ec0c0bf-cc68-45cd-9b57-d689028f0f7a,1ec0c0bf-cc68-45cd-9b57-d689028f0f7a,Fundamentals of Computer Networks,Fundamentals of Computer Networks,03.1,2,False,False,Explain why a firewall improves network security.
6,Explain how cyber security protects a network.,2,0.592461,749f4415-a2ea-4267-82f6-a73718580243,749f4415-a2ea-4267-82f6-a73718580243,Cyber Security,Methods to Detect and Prevent Cyber Security T...,05.3,9,False,False,The network manager of a new computer games co...
7,Explain how cyber security protects a network.,3,0.581240,66190430-2b6c-4b4d-af13-8286e9619b21,66190430-2b6c-4b4d-af13-8286e9619b21,Fundamentals of Computer Networks,Fundamentals of Computer Networks,12,2,False,False,Describe how encryption can make the transmiss...
8,Explain how cyber security protects a network.,4,0.570810,2e0053e0-373d-4246-b62c-3665a38f9029,2e0053e0-373d-4246-b62c-3665a38f9029,Cyber Security,Methods to Detect and Prevent Cyber Security T...,04.2,2,False,False,Describe one security measure that could be us...
9,Explain how cyber security protects a network.,5,0.568882,2887db8f-b580-46fc-93c7-1ea7d6e74e9f,2887db8f-b580-46fc-93c7-1ea7d6e74e9f,Cyber Security,Fundamentals of Cyber Security,03.1,2,False,False,Define the term cyber security.


## 21. Self-retrieval integrity check

An exact question-text query should normally retrieve its own point near the top.
This checks storage and query consistency; it is not a complete retrieval-quality evaluation.


In [70]:
SELF_RETRIEVAL_SAMPLE_SIZE = 12
SELF_RETRIEVAL_LIMIT = 3


self_retrieval_rows = []

if qdrant_point_count == 0:
    print(
        "Self-retrieval test skipped because "
        "Qdrant contains no points."
    )

else:
    sample_size = min(
        SELF_RETRIEVAL_SAMPLE_SIZE,
        len(eligible_df),
    )

    sample_df = eligible_df.sample(
        n=sample_size,
        random_state=RANDOM_SEED,
    )

    for _, row in sample_df.iterrows():
        results_df = semantic_search(
            row["question_text"],
            limit=SELF_RETRIEVAL_LIMIT,
        )

        returned_ids = (
            results_df["question_id"]
            .astype(str)
            .tolist()
            if not results_df.empty
            else []
        )

        expected_id = str(
            row["question_id"]
        )

        found_rank = (
            returned_ids.index(expected_id) + 1
            if expected_id in returned_ids
            else None
        )

        self_retrieval_rows.append(
            {
                "question_id": expected_id,
                "question_number": (
                    row["question_number"]
                ),
                "topic": row["topic_name"],
                "found_in_top_k": (
                    found_rank is not None
                ),
                "found_rank": found_rank,
                "top_score": (
                    float(
                        results_df.iloc[0][
                            "score"
                        ]
                    )
                    if not results_df.empty
                    else None
                ),
            }
        )


self_retrieval_df = pd.DataFrame(
    self_retrieval_rows
)

display(self_retrieval_df)

if not self_retrieval_df.empty:
    self_retrieval_pass_rate = float(
        self_retrieval_df[
            "found_in_top_k"
        ].mean()
    )

    print(
        f"Self-retrieval top-{SELF_RETRIEVAL_LIMIT} "
        f"pass rate: "
        f"{self_retrieval_pass_rate:.1%}"
    )


,question_id,question_number,topic,found_in_top_k,found_rank,top_score
0,634abc99-a536-4dac-b122-56bfd47ace3a,04.2,Computer Systems,True,1.0,0.781221
1,5a10626a-dd80-4f60-a736-f2e8dd46219d,10,Programming,True,1.0,0.909130
2,c326fd67-9507-4ff8-947c-bebb9ae9f140,01.1,Fundamentals of Algorithms,True,3.0,0.783928
3,fa6be55d-d2e3-49f8-8ac1-1e69f4c58dbb,06.1,Computer Systems,True,1.0,0.798896
4,f2563457-34cd-4fd5-91d3-4784aede3faf,06.3,Programming,True,2.0,0.794406
5,c9d5bb1e-d2c5-4228-b455-ef2e08a5f97f,02.1,Programming,True,1.0,0.855957
6,59afa777-1b98-4646-9412-d7e7ba6092da,01.6,Relational Databases and Structured Query Lang...,True,1.0,0.881411
7,f452914d-258d-4348-a88a-3b1a68874545,04.3,Programming,True,1.0,0.732243
8,190deaf5-d29b-4c8b-a8f7-ba1b26c4720a,06.1,Fundamentals of Data Representation,True,2.0,0.671269
9,2051ad2f-2c8c-4adb-b749-70cbc303cb2c,16.2,Computer Systems,True,1.0,0.780277


Self-retrieval top-3 pass rate: 91.7%


## 22. Final Notebook 04 readiness checks


In [71]:
full_committed_run = (
    RUN_MODE == "full"
    and COMMIT_TO_QDRANT
)

final_checks = {
    "eligible_questions_exist": (
        len(eligible_df) > 0
    ),
    "vector_size_is_384": (
        VECTOR_SIZE
        == EXPECTED_VECTOR_SIZE
    ),
    "no_indexing_failures": (
        len(failed_records) == 0
    ),
    "qdrant_matches_eligible_count": (
        qdrant_point_count
        == len(eligible_df)
    ),
    "postgres_index_matches_eligible_count": (
        postgres_indexed_count
        == len(eligible_df)
    ),
    "question_status_matches_eligible_count": (
        indexed_question_count
        == len(eligible_df)
    ),
    "no_questions_waiting_for_indexing": (
        ready_for_indexing_count == 0
    ),
}


final_checks_df = pd.DataFrame(
    [
        {
            "check": key,
            "passed": bool(value),
        }
        for key, value
        in final_checks.items()
    ]
)

display(final_checks_df)


notebook_04_complete = (
    full_committed_run
    and all(
        final_checks.values()
    )
)

print(
    f"Full committed run:    "
    f"{full_committed_run}"
)

print(
    f"Notebook 04 complete:  "
    f"{notebook_04_complete}"
)


if RUN_MODE == "pilot":
    print(
        "\nNext: set RUN_MODE='full' and keep "
        "COMMIT_TO_QDRANT=False for the full dry run."
    )

elif not COMMIT_TO_QDRANT:
    print(
        "\nNext: keep RUN_MODE='full' and set "
        "COMMIT_TO_QDRANT=True."
    )


,check,passed
0,eligible_questions_exist,True
1,vector_size_is_384,True
2,no_indexing_failures,True
3,qdrant_matches_eligible_count,True
4,postgres_index_matches_eligible_count,True
5,question_status_matches_eligible_count,True
6,no_questions_waiting_for_indexing,True


Full committed run:    True
Notebook 04 complete:  True


## 23. Export indexing reports


In [72]:
timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

run_label = (
    f"{RUN_MODE}_"
    f"{'committed' if COMMIT_TO_QDRANT else 'dry_run'}"
)

eligibility_path = (
    OUTPUT_DIR
    / f"agent2_minilm_eligibility_{run_label}_{timestamp}.csv"
)

decisions_path = (
    OUTPUT_DIR
    / f"agent2_minilm_indexing_decisions_{run_label}_{timestamp}.csv"
)

failures_path = (
    OUTPUT_DIR
    / f"agent2_minilm_indexing_failures_{run_label}_{timestamp}.csv"
)

summary_path = (
    OUTPUT_DIR
    / f"agent2_minilm_qdrant_summary_{run_label}_{timestamp}.json"
)


eligible_df[
    [
        "question_id",
        "question_uid",
        "pair_key",
        "topic_number",
        "topic_name",
        "subtopic_code",
        "subtopic_name",
        "question_number",
        "marks",
        "review_status",
        "embedding_status",
        "embedding_token_count",
        "embedding_was_truncated",
        "embedding_text_hash",
    ]
].to_csv(
    eligibility_path,
    index=False,
)

selected_df[
    [
        "question_id",
        "question_uid",
        "pair_key",
        "question_number",
        "needs_indexing",
        "indexing_reasons",
        "embedding_text_hash",
    ]
].to_csv(
    decisions_path,
    index=False,
)

pd.DataFrame(
    failed_records
).to_csv(
    failures_path,
    index=False,
)


summary_payload = {
    "generated_at_utc": utc_now(),
    "indexer_version": INDEXER_VERSION,
    "embedding_model": MODEL_NAME,
    "embedding_dimension": VECTOR_SIZE,
    "collection_name": COLLECTION_NAME,
    "run_mode": RUN_MODE,
    "committed": COMMIT_TO_QDRANT,
    "eligible_count": len(eligible_df),
    "selected_count": len(selected_df),
    "pending_count": len(pending_df),
    "skipped_count": len(skipped_df),
    "successful_count": len(
        successful_question_ids
    ),
    "failed_count": len(
        failed_records
    ),
    "stale_deleted_count": (
        stale_deleted_count
    ),
    "qdrant_point_count": (
        qdrant_point_count
    ),
    "postgres_indexed_count": (
        postgres_indexed_count
    ),
    "indexed_question_count": (
        indexed_question_count
    ),
    "ready_for_indexing_count": (
        ready_for_indexing_count
    ),
    "notebook_04_complete": (
        notebook_04_complete
    ),
    "final_checks": final_checks,
    "output_files": {
        "eligibility": str(
            eligibility_path.resolve()
        ),
        "decisions": str(
            decisions_path.resolve()
        ),
        "failures": str(
            failures_path.resolve()
        ),
    },
}


summary_path.write_text(
    json.dumps(
        strict_json_value(
            summary_payload
        ),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("Saved:")
print(eligibility_path)
print(decisions_path)
print(failures_path)
print(summary_path)


Saved:
C:\Users\hp\EDTECH\Agent2\OUTPUT\agent2_minilm_eligibility_full_committed_20260804_213454.csv
C:\Users\hp\EDTECH\Agent2\OUTPUT\agent2_minilm_indexing_decisions_full_committed_20260804_213454.csv
C:\Users\hp\EDTECH\Agent2\OUTPUT\agent2_minilm_indexing_failures_full_committed_20260804_213454.csv
C:\Users\hp\EDTECH\Agent2\OUTPUT\agent2_minilm_qdrant_summary_full_committed_20260804_213454.json


# Notebook 04 completion criteria

A successful full committed run should show:

```text
eligible_questions                  = 820
qdrant_point_count                  = 820
postgres_vector_index_count         = 820
questions_embedding_status_indexed  = 820
questions_ready_for_indexing        = 0
indexing failures                   = 0
Notebook 04 complete                = True
```

## Next step

```text
Notebook 05 — Agent 1 Topic → Agent 2 Assessment Retrieval
```

Notebook 05 will:

- accept Agent 1 topic/subtopic output;
- generate a MiniLM query vector;
- apply topic, paper, language, marks, code, and visual filters;
- retrieve the best questions from Qdrant;
- fetch complete mark-scheme guidance from PostgreSQL;
- rerank and return the final assessment package.
